# Queueing theory - M/M/1 using Dask

A single M/M/1 queue simulated many times in parallel. Every replication simulates 100,000 customers with its own independent random seeds. The replications show how much a single simulation result can vary, and their average approaches the analytical (steady-state) solution.

The calculations are distributed with [Dask](https://docs.dask.org/en/stable/deploying.html):
* locally, a `LocalCluster` starts several worker *processes* on your computer (processes rather than threads, because the simulation is pure Python and threads would be blocked by Python's global interpreter lock);
* to upscale, the same code can connect to a larger cluster (e.g. a HPC or cloud cluster) by replacing the `LocalCluster` (see section 2).

NB: Dask is an optional dependency. Install it in your environment with `pip install "dask[distributed,diagnostics]"` (or `pip install -e ".[dask]"` from the OpenQTSim folder).

In [1]:
# OpenQTSim parallel engine
import os
import openqtsim
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dask.distributed import Client, LocalCluster

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [ ]:
A = "M"                  # distribution of interarrival times
S = "M"                  # distribution of service times

Lambda = 8               # nr of arrivals per hour
Mu = 9                   # nr of exits (services completed) per hour

nr_arr = 100_000         # number of customers per replication

nr_replications = 24     # independent replications (Dask tasks)
seed = 2024              # makes all replications reproducible
n_workers = os.cpu_count()  # number of local Dask worker processes

#### 1. Create task list
Each task is one replication with its own independent arrival and service seeds.

In [3]:
tasks = create_tasks(A, S, [1], [Lambda / Mu], nr_arr, mu=Mu, nr_replications=nr_replications, seed=seed)

print(f"{len(tasks)} replications of {nr_arr:,} customers each ({len(tasks) * nr_arr:,} customers in total)")
pd.DataFrame(tasks).head()

24 replications of 100,000 customers each (2,400,000 customers in total)


,A,S,c,nr_arr,lam,mu,replication,arrival_seed,service_seed
0,M,M,1,100000,8.0,9,0,2855298535,925030728
1,M,M,1,100000,8.0,9,1,1146676384,1811149567
2,M,M,1,100000,8.0,9,2,524768821,2577401382
3,M,M,1,100000,8.0,9,3,4067285479,2609318523
4,M,M,1,100000,8.0,9,4,2651666590,174624609


#### 2. Start a Dask cluster
The client shows a link to the Dask dashboard (open it in your browser), where you can follow the workers and the progress of the tasks while they run.

To upscale beyond your own computer, replace the `LocalCluster` by a larger cluster; the rest of the notebook stays the same. For example on a SLURM cluster such as DelftBlue (with `pip install dask-jobqueue`):

```python
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(cores=8, processes=8, memory="16GB", walltime="01:00:00")
cluster.scale(jobs=10)  # 10 jobs x 8 worker processes = 80 workers
```

or connect to a running scheduler with `client = Client("tcp://scheduler-address:8786")`.

In [4]:
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 12,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57010,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:57037,Total threads: 1
Dashboard: http://127.0.0.1:57038/status,Memory: 1.33 GiB
Nanny: tcp://127.0.0.1:57013,


#### 3. Run the replications in parallel
`statistics_worker` returns the same statistics as `Simulation.get_stats()` for every replication.

In [5]:
%%time
results = pd.DataFrame(run_tasks(tasks, client, func=statistics_worker))
measures = ["IAT", "ST", "Rho_server", "P_0", "L_s", "L_q", "W_s", "W_q", "waiting_factor"]
results[["replication"] + measures].round(4)

CPU times: user 684 ms, sys: 316 ms, total: 1e+03 ms
Wall time: 13.1 s


,replication,IAT,ST,Rho_server,P_0,L_s,L_q,W_s,W_q,waiting_factor
0,0,0.1253,0.1114,0.8894,0.1106,7.9974,7.1080,1.0018,0.8904,7.9918
1,1,0.1248,0.1108,0.8873,0.1127,7.6484,6.7610,0.9549,0.8441,7.6197
2,2,0.1259,0.1110,0.8815,0.1185,7.3527,6.4713,0.9255,0.8146,7.3415
3,3,0.1250,0.1114,0.8915,0.1085,8.6855,7.7940,1.0854,0.9740,8.7424
4,4,0.1252,0.1113,0.8891,0.1109,7.9603,7.0712,0.9968,0.8854,7.9535
5,5,0.1245,0.1112,0.8933,0.1067,8.5707,7.6775,1.0672,0.9560,8.5948
6,6,0.1255,0.1106,0.8813,0.1187,7.4263,6.5450,0.9322,0.8216,7.4261
7,7,0.1250,0.1114,0.8916,0.1084,7.6502,6.7586,0.9560,0.8446,7.5800
8,8,0.1247,0.1113,0.8922,0.1078,8.4871,7.5949,1.0585,0.9473,8.5126
9,9,0.1250,0.1103,0.8830,0.1170,7.4620,6.5790,0.9324,0.8221,7.4506


#### 4. Compare with the analytical solution

For an M/M/1 queue with $\rho = \lambda / \mu < 1$: $P_0 = 1-\rho$, $L_s = \rho/(1-\rho)$, $L_q = \rho^2/(1-\rho)$, $W_s = 1/(\mu-\lambda)$, $W_q = \rho/(\mu-\lambda)$ and $W_q / E[ST] = \rho / (1-\rho)$.

The standard error of the average over all replications is the standard deviation between the replications divided by $\sqrt{\text{nr\_replications}}$.

In [6]:
rho = Lambda / Mu
analytical = pd.Series({
    "IAT": 1 / Lambda, "ST": 1 / Mu, "Rho_server": rho, "P_0": 1 - rho,
    "L_s": rho / (1 - rho), "L_q": rho**2 / (1 - rho),
    "W_s": 1 / (Mu - Lambda), "W_q": rho / (Mu - Lambda), "waiting_factor": rho / (1 - rho),
})

comparison = pd.DataFrame({
    "analytical": analytical,
    "average of replications": results[measures].mean(),
    "standard error": results[measures].std() / np.sqrt(len(results)),
    "min replication": results[measures].min(),
    "max replication": results[measures].max(),
})
comparison["difference (%)"] = 100 * (comparison["average of replications"] / comparison["analytical"] - 1)
comparison.round(4)

,analytical,average of replications,standard error,min replication,max replication,difference (%)
IAT,0.1250,0.1251,0.0001,0.1245,0.1259,0.0813
ST,0.1111,0.1111,0.0001,0.1103,0.1118,-0.0170
Rho_server,0.8889,0.8880,0.0008,0.8812,0.8933,-0.1054
P_0,0.1111,0.1120,0.0008,0.1067,0.1188,0.8435
L_s,8.0000,7.9230,0.0937,7.1619,8.8787,-0.9623
L_q,7.1111,7.0351,0.0930,6.2790,7.9855,-1.0694
W_s,1.0000,0.9912,0.0114,0.8967,1.1070,-0.8813
W_q,0.8889,0.8801,0.0114,0.7861,0.9957,-0.9894
waiting_factor,8.0000,7.9211,0.0987,7.1113,8.9411,-0.9860


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results["replication"] + 1, y=results["W_q"], mode="markers", name="W_q per replication",
))
fig.add_trace(go.Scatter(
    x=results["replication"] + 1, y=results["W_q"].expanding().mean(), mode="lines+markers",
    name="running average over replications",
))
fig.add_hline(y=analytical["W_q"], line_dash="dash", line_color="black",
              annotation_text=f"analytical W_q = {analytical['W_q']:.4f}", annotation_position="top left")
fig.update_layout(
    title=f"M/M/1 (ρ = {rho:.2f}): average waiting time per replication of {nr_arr:,} customers",
    xaxis_title="replication", yaxis_title="W_q [hours]", height=500, template="plotly_white",
)
fig.show()

At a high utilisation ($\rho \approx 0.89$) successive waiting times are strongly correlated, so a single replication of 100,000 customers can still deviate several percent from the analytical solution. Averaging independent replications, which Dask runs in parallel, steadily reduces this uncertainty.

#### 6. Close the Dask cluster

In [8]:
client.close()
cluster.close()